# 15 Analyze Mutants

Use an LLM to generate concise, human-interpretable explanations for prioritized mutation units in a backbone enzyme mutagenesis dataset.

## Workflow

This notebook:
- loads the selected mutations CSV
- loads residue-level structural context and binding-pocket residue context
- loads summarized binding-pocket metrics for the selected backbone enzyme
- optionally includes prior binding-pocket LLM context for the same enzyme
- groups single and multi-mutation variants into analysis units
- generates a final CSV with one explanation per unit

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))


## Imports

In [2]:
import importlib
import agentic_protein_design.steps.analyze_mutants as am

am = importlib.reload(am)
from agentic_protein_design.core import apply_notebook_markdown_style

default_user_inputs = am.default_user_inputs
default_input_paths = am.default_input_paths
setup_data_root = am.setup_data_root
run_analyze_mutants_step = am.run_analyze_mutants_step
reflect_and_regenerate_mutant_explanations = am.reflect_and_regenerate_mutant_explanations
save_mutant_explanations_csv = am.save_mutant_explanations_csv
save_llm_analysis = am.save_llm_analysis
MUTANT_ANALYSIS_REFLECTION_PROMPT = am.MUTANT_ANALYSIS_REFLECTION_PROMPT

apply_notebook_markdown_style(font_size_px=14, line_height=1.4)


## User Inputs

Set the backbone enzyme, ligand used for structural analyses, and the required CSV filenames relative to the selected data root.

In [3]:
root_key = "examples"
existing_thread_key = None
persist = True

data_root, _ = setup_data_root(root_key)

user_inputs = default_user_inputs()
user_inputs.update({
    "enzyme_name": "ET096",
    "ligand_name": "S82",
    "focus_question": (
        "Explain how prioritized mutations in ET096 likely affect activity, selectivity, stability, solubility, expression and pocket behavior, using the S82 structural context plus any prior binding-pocket and literature context.",
        "For UPOs, methionine residues in the binding pocket have been known to improve peroxide tolerance."
    ),
    "llm_model": "gpt-5.2",
    "llm_temperature": 0.2,
    "llm_max_rows": 200,
    "display_llm_output": True,
    "display_max_height": "640px",
    "binding_pocket_context_thread_key": "",  # optional
    "literature_context_thread_key": "literature_review_d762a72ec7f04bec9b66ccd3aac21b91",  # optional
})

input_paths = default_input_paths(data_root)
input_paths.update({
    "selected_mutations_csv": "mutagenesis_proposal/SelectedMuts_ET096_mutagenesis_wEthylBenzene&Purified_2026-02-05.csv",
    "residue_structure_csv": "pdb/structure_csv/ET096_S82_backbone.csv",
    "binding_residues_csv": "pdb/structure_csv/ET096_S82_backbone_bindingpocket.csv",
    "binding_summary_csv": "pdb/bindingpocket_analysis.csv",
})

user_inputs, input_paths


({'enzyme_name': 'ET096',
  'ligand_name': 'S82',
  'focus_question': ('Explain how prioritized mutations in ET096 likely affect activity, selectivity, stability, solubility, expression and pocket behavior, using the S82 structural context plus any prior binding-pocket and literature context.',
   'For UPOs, methionine residues in the binding pocket have been known to improve peroxide tolerance.'),
  'llm_model': 'gpt-5.2',
  'llm_temperature': 0.2,
  'llm_max_rows': 200,
  'display_llm_output': True,
  'display_max_height': '640px',
  'display_compact_markdown': False,
  'binding_pocket_context_thread_key': '',
  'literature_context_thread_key': 'literature_review_d762a72ec7f04bec9b66ccd3aac21b91'},
 {'selected_mutations_csv': 'mutagenesis_proposal/SelectedMuts_ET096_mutagenesis_wEthylBenzene&Purified_2026-02-05.csv',
  'residue_structure_csv': 'pdb/structure_csv/ET096_S82_backbone.csv',
  'binding_residues_csv': 'pdb/structure_csv/ET096_S82_backbone_bindingpocket.csv',
  'binding_sum

## Run Mutant Analysis

In [4]:
result = run_analyze_mutants_step(
    root_key=root_key,
    user_inputs=user_inputs,
    input_paths=input_paths,
    existing_thread_key=existing_thread_key,
    persist=persist,
)

{
    "thread_id": result["thread_id"],
    "step_processed_dir": str(result["step_processed_dir"]),
    "explanations_csv_path": str(result["explanations_csv_path"]),
    "llm_analysis_path": str(result["llm_analysis_path"]),
}

### Mutant Analysis LLM Call

<details><summary>Prompt</summary>

```text
You are a protein engineer analyzing a mutagenesis dataset for a backbone enzyme.

Goal:
Generate concise, human-interpretable mechanistic explanations for prioritized mutation units.
These units are already grouped for you as:
- single-position groups from single mutants,
- single substitutions,
- clusters of multi-mutation mutants.

Use the provided assay data, residue-level structural context, binding-pocket membership, ligand distances,
and backbone binding-pocket summary metrics to infer likely effects on activity, selectivity, expression,
and binding-pocket behavior.

Output contract (strict):
- Return ONLY a JSON array.
- Return one object per provided analysis unit.
- Each object must contain:
  - row_index: integer copied from the provided analysis unit row_index
  - Description of effect: one sentence, compact but specific
- Do not return markdown, code fences, or extra prose.

Rules:
- Ground all reasoning in the provided context, as well as any background knowledge of the amino acid substitutions, enzyme and reaction.
- If a residue is absent from the binding-pocket residue table, treat it as likely outside the defined binding pocket/tunnel region.
- For single-position groups, describe only why that residue position is important or sensitive in the enzyme context.
- Do not mention any specific substitution (for example Val -> Thr) in the position-level explanation, even if only one substitution is available for that position.
- For single substitutions, describe the likely effect of that exact amino-acid substitution (for example Ala -> Phe), with emphasis on the chemistry/size/polarity change introduced by the substitution itself rather than repeating the generic position-level effect.
- For multi-mutation clusters, describe the shared or net effect of the cluster.
- Mention uncertainty briefly when the evidence is weak or conflicting.

PROJECT CONTEXT
- Backbone enzyme: ET096
- Ligand / analysis context: S82
- Objective: ('Explain how prioritized mutations in ET096 likely affect activity, selectivity, stability, solubility, expression and pocket behavior, using the S82 structural context plus any prior binding-pocket and literature context.', 'For UPOs, methionine residues in the binding pocket have been known to improve peroxide tolerance.')
- Selected mutants loaded: 140 rows
- Analysis units to explain: 112 rows
- Key mutant assay/property columns present: foldchange_NBD_activity_25C, foldchange_ABTS_activity_25C, FC_NBD_1mM_purified, FC_ABTS_1mM_purified, FC_protein_yield, Foldchange_Unk_Area_Norm_Subtracted, Ketone%, Alcohol%, Total%
- Residue-level structural columns available: res_num, res_name, res, aa_polarity, kd_hydro, hw_polarity, aa_vol, dist_res_to_ligand_reactive_center, min_dist_res_to_ligand
- Backbone binding-pocket summary columns highlighted: struct_name, num_pocket_res_ali, num_pocket_res<6, reactive_center_distance, median_dist_res_to_ligand_reactive_center, median_min_dist_res_to_ligand, mean_volume (proximal), mean_volume (distal), kd_weighted (proximal), kd_weighted (distal), hw_weighted (proximal), hw_weighted (distal), charged_fraction (proximal), charged_fraction (distal), polar_fraction (proximal), polar_fraction (distal)
- Important interpretation rule: residues absent from the binding-pocket residue table should be treated as outside the defined pocket/tunnel region and typically farther from the ligand than listed pocket residues.
```
</details>

#### Response

(Final explanation table preview shown below.)

### Mutant Explanation Table

| Mutant(s)                                                                                    | Description of effect                                                                                                                                                                                                                                                                                                                                          |
|:---------------------------------------------------------------------------------------------|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| V138T*                                                                                       | Position 138 is outside the pocket and likely affects stability or long-range packing, giving modest activity/product shifts consistent with an indirect structural effect.                                                                                                                                                                                    |
|                                                                                              | V138T introduces a polar hydroxyl outside the pocket, likely increasing local hydration and potentially stabilizing structure, giving modest activity and product gains consistent with an indirect effect.                                                                                                                                                    |
| H143Q*; H143T*                                                                               | Position 143 is outside the pocket and appears to tune electrostatics/protonation networks that influence the peroxygenative:peroxidative balance, with broad effects across multi-mutants indicating strong epistasis.                                                                                                                                        |
|                                                                                              | H143Q removes positive charge and metal-like protonation behavior at a non-pocket site, likely weakening electrostatic/proton-coupled steps that support oxygen transfer (lower NBD) while favoring ABTS oxidation via altered redox/proton balance.                                                                                                           |
|                                                                                              | H143T removes histidine protonation capability at a non-pocket site, likely shifting local hydrogen-bonding/electrostatics to broadly enhance NBD in multi-mutants while variably affecting ABTS, consistent with an indirect network effect.                                                                                                                  |
| A167E*                                                                                       | Position 167 is outside the pocket and likely impacts stability or surface electrostatics, where introducing charge can shift activity/selectivity indirectly (evidence currently sparse).                                                                                                                                                                     |
|                                                                                              | A167E introduces a negative charge at a non-pocket site, likely improving solubility or altering electrostatic networks to boost NBD activity but strongly reducing ABTS, consistent with shifting away from peroxidative electron-transfer pathways.                                                                                                          |
| A171F; A171I; A171L; A171V                                                                   | Position 171 is a distal pocket-lining site (~7 Å min distance) where changing side-chain bulk/hydrophobicity likely remodels the channel wall to favor productive substrate positioning for oxygen transfer while variably impacting peroxidase-like turnover.                                                                                                |
|                                                                                              | A171F introduces a much bulkier aromatic side chain in the distal pocket, likely narrowing/reshaping the channel to favor a binding pose that boosts NBD oxygen-transfer activity while disfavoring ABTS-type peroxidative turnover.                                                                                                                           |
|                                                                                              | A171I increases hydrophobic bulk at a distal pocket wall, likely improving substrate packing/orientation for both oxygen-transfer and peroxidase readouts by stabilizing productive binding without overly constricting the channel.                                                                                                                           |
|                                                                                              | A171L adds hydrophobic volume in the distal pocket that likely tightens the channel and biases substrate positioning toward oxygen transfer (higher NBD) while partially suppressing peroxidative ABTS activity via altered access/pose.                                                                                                                       |
|                                                                                              | A171V modestly increases hydrophobic packing at the distal pocket wall, likely providing a balanced channel reshaping that improves both NBD and ABTS activities by stabilizing binding with limited steric penalty.                                                                                                                                           |
| L174F                                                                                        | Position 174 is a close pocket contact (~3.4 Å) that likely acts as a tight steric gate controlling substrate approach angle to the heme-oxo, making it highly influential on activity/selectivity tradeoffs.                                                                                                                                                  |
|                                                                                              | L174F adds a bulky aromatic side chain at a very close pocket contact (~3.4 Å), likely creating steric crowding that impairs productive oxygen-transfer binding (lower NBD) while still allowing or even favoring ABTS oxidation through altered access dynamics.                                                                                              |
| S182A; S182C; S182L; S182M; S182V                                                            | Position 182 is a distal pocket residue (~7 Å min distance) that likely modulates channel polarity and packing near the exit/secondary cavity, affecting substrate binding dynamics and the peroxygenative:peroxidative activity ratio.                                                                                                                        |
|                                                                                              | S182A removes a distal-pocket hydroxyl, likely reducing local polarity and H-bonding to modestly favor hydrophobic substrate binding and slightly increase NBD while leaving ABTS largely unchanged.                                                                                                                                                           |
|                                                                                              | S182C replaces a hydroxyl with a softer thiol-like side chain at the distal pocket, likely altering local polarizability and packing to moderately improve both NBD and ABTS activities without strong steric effects.                                                                                                                                         |
|                                                                                              | S182L introduces a larger hydrophobic side chain at the distal pocket, likely tightening the channel and improving substrate residence/pose to increase both NBD and ABTS activities.                                                                                                                                                                          |
|                                                                                              | S182M adds a larger, polarizable thioether in the distal pocket, which can enhance hydrophobic packing and (in UPOs) potentially improve oxidative robustness, boosting NBD while leaving ABTS near neutral.                                                                                                                                                   |
|                                                                                              | S182V increases hydrophobicity with a branched side chain at the distal pocket, likely shifting binding toward poses that enhance ABTS oxidation but reduce NBD oxygen-transfer efficiency, consistent with a chemoselectivity tradeoff.                                                                                                                       |
| E197K*                                                                                       | Position 197 is outside the pocket and likely modulates surface charge networks that affect folding/solubility or long-range electrostatics, indirectly improving activity.                                                                                                                                                                                    |
|                                                                                              | E197K flips a surface charge from negative to positive outside the pocket, likely improving expression/solubility and altering long-range electrostatics to increase activity.                                                                                                                                                                                 |
| H208D                                                                                        | Position 208 lies outside the defined pocket and likely influences catalysis indirectly via electrostatics/proton networks or global stability, which can change peroxide utilization efficiency and apparent activity without direct ligand contact.                                                                                                          |
|                                                                                              | H208D replaces a potentially protonatable residue with a fixed negative charge outside the pocket, likely rewiring local electrostatics/proton handling to improve apparent activity and yield while slightly reducing purified ABTS performance, consistent with an indirect mechanistic effect.                                                              |
| Y212K; Y212T                                                                                 | Position 212 is outside the defined pocket and appears to be a strong lever on expression/yield and overall catalytic output, likely by altering surface charge/solubility or long-range electrostatics that couple to peroxide activation and electron-transfer (peroxidative) pathways.                                                                      |
|                                                                                              | Y212K introduces a strong positive charge on a non-pocket surface position, likely improving expression/solubility and altering long-range electrostatics to boost both NBD and ABTS, though the highly negative 'Unk area' signal suggests possible assay interference or altered side-reaction chemistry.                                                    |
|                                                                                              | Y212T removes an aromatic ring and reduces side-chain size/polarity at a non-pocket position, likely improving folding/secretion (higher yield) while giving moderate activity gains via indirect electrostatic/dynamic effects.                                                                                                                               |
| S214P*                                                                                       | Position 214 is outside the pocket and likely affects local backbone rigidity (proline sensitivity) and thus global stability/trafficking, indirectly influencing activity and product formation.                                                                                                                                                              |
|                                                                                              | S214P introduces a rigid proline outside the pocket, likely stabilizing a loop/turn and improving overall folding or dynamics, correlating with increased activity and product formation.                                                                                                                                                                      |
| S217P*                                                                                       | Position 217 is outside the pocket and likely acts as a loop/hinge site where increased rigidity can alter access-channel dynamics or stability, strongly impacting ABTS activity in the available mutants.                                                                                                                                                    |
|                                                                                              | S217P introduces proline-mediated rigidity outside the pocket, likely altering loop dynamics that control access or stability and strongly boosting ABTS oxidation while only modestly affecting NBD.                                                                                                                                                          |
| F220L                                                                                        | Position 220 is a pocket residue (~6.4 Å min distance) that likely contributes aromatic/hydrophobic packing in the channel and helps pre-organize substrates for oxygen transfer, so perturbations here can strongly shift activity and chemoselectivity.                                                                                                      |
|                                                                                              | F220L removes an aromatic ring from a pocket-lining position, likely weakening π/stacking and loosening hydrophobic packing so substrates bind less productively for oxygen transfer (strong NBD drop) while ABTS oxidation can increase due to easier access or reduced steric gating.                                                                        |
| F223L*                                                                                       | Position 223 is a close pocket residue (~3.8 Å min distance) that likely forms a key hydrophobic/aromatic wall controlling substrate approach and residence time, making it a strong lever for ABTS/peroxygenation balance.                                                                                                                                    |
|                                                                                              | F223L removes an aromatic ring at a close pocket contact (~3.8 Å), likely enlarging/softening a gate to increase ABTS oxidation (greater access/turnover) while giving smaller gains in NBD due to less precise substrate pre-organization.                                                                                                                    |
| Q236L*                                                                                       | Position 236 is outside the pocket but highly epistatic in multi-mutants, consistent with a role in stability/solubility or long-range electrostatics that modulate overall catalytic output rather than direct binding.                                                                                                                                       |
|                                                                                              | Q236L removes a polar amide outside the pocket, likely increasing local hydrophobic packing and stability, indirectly supporting higher activity and yield across many multi-mutants.                                                                                                                                                                          |
| S237V*                                                                                       | Position 237 is outside the pocket and likely influences local flexibility/packing near the C-terminus, indirectly affecting activity with moderate evidence.                                                                                                                                                                                                  |
|                                                                                              | S237V increases hydrophobicity outside the pocket, likely stabilizing local structure and reducing flexibility, indirectly supporting higher activity in multi-mutant backgrounds.                                                                                                                                                                             |
| R239E*                                                                                       | Position 239 is outside the pocket and likely affects surface electrostatics and possibly secretion/solubility, indirectly modulating activity in multi-mutant contexts.                                                                                                                                                                                       |
|                                                                                              | R239E reverses charge outside the pocket, likely rewiring surface salt-bridge networks and solubility; in multi-mutants this can improve activity but may also risk misfolding depending on context (uncertain without isolated single-mutant data).                                                                                                           |
| A240Q*                                                                                       | Position 240 is outside the pocket and likely influences local packing or polarity near the protein surface, indirectly affecting activity when combined with other mutations.                                                                                                                                                                                 |
|                                                                                              | A240Q adds a polar amide at a non-pocket position, likely increasing local hydrogen-bonding and solubility and indirectly supporting higher activity in multi-mutant backgrounds.                                                                                                                                                                              |
| I241S*                                                                                       | Position 241 is outside the pocket and likely affects local hydrophobic core/surface balance, indirectly tuning stability and catalytic output in multi-mutants.                                                                                                                                                                                               |
|                                                                                              | I241S introduces a polar hydroxyl at a non-pocket hydrophobic site, likely increasing local hydration/solubility and altering packing near the C-terminus, indirectly supporting higher activity in the multi-mutant context.                                                                                                                                  |
| E242S*                                                                                       | Position 242 is outside the pocket and likely alters surface charge/polarity, indirectly affecting folding/solubility and thus apparent activity.                                                                                                                                                                                                              |
|                                                                                              | E242S removes a negative charge outside the pocket, likely reducing local electrostatic frustration and improving folding/solubility, indirectly supporting higher activity.                                                                                                                                                                                   |
| L243C*                                                                                       | Position 243 is outside the pocket and likely impacts local packing near the C-terminus, indirectly influencing activity with limited direct evidence.                                                                                                                                                                                                         |
|                                                                                              | L243C introduces a smaller, more polarizable side chain outside the pocket, which may subtly alter local packing or enable new stabilizing interactions, indirectly affecting activity with moderate uncertainty.                                                                                                                                              |
| S29A*; S29P*                                                                                 | Position 29 is outside the defined pocket yet repeatedly associates with large activity/yield shifts in multi-mutants, suggesting it is a structural/trafficking hotspot (e.g., local folding or secretion efficiency) that indirectly amplifies catalytic performance.                                                                                        |
|                                                                                              | S29A removes a polar hydroxyl outside the pocket, likely reducing local polarity and modestly improving stability/packing, giving moderate activity and product gains.                                                                                                                                                                                         |
|                                                                                              | S29P introduces backbone rigidity outside the pocket, likely stabilizing a local structural element important for folding/trafficking and thereby enabling large activity/yield gains across many multi-mutants.                                                                                                                                               |
| L38M*                                                                                        | Position 38 is a distal pocket residue (~7 Å min distance) that likely shapes the access channel and influences peroxidase vs peroxygenase readouts, with strong sensitivity in ABTS suggesting altered substrate access or electron-transfer coupling.                                                                                                        |
|                                                                                              | L38M adds a more polarizable thioether at a distal pocket wall (~7 Å min distance), likely reshaping the access channel to strongly enhance ABTS oxidation and modestly improve NBD, consistent with altered substrate access and binding dynamics.                                                                                                            |
| S41A*                                                                                        | Position 41 is outside the pocket but strongly enriched among high-activity multi-mutants, consistent with a role in protein stability/expression or global dynamics that indirectly increases turnover and product formation.                                                                                                                                 |
|                                                                                              | S41A removes a polar hydroxyl outside the pocket, likely improving local packing and reducing misfolding/aggregation, which broadly increases apparent activity and product formation in multi-mutant backgrounds.                                                                                                                                             |
| G57A*; G57L*                                                                                 | Position 57 is outside the pocket and likely affects local backbone flexibility (glycine sensitivity) and folding/trafficking, which can indirectly raise apparent activity and yields across many multi-mutants.                                                                                                                                              |
|                                                                                              | G57A replaces glycine with a small side chain outside the pocket, likely stabilizing local backbone conformations and improving folding/trafficking, which correlates with higher apparent activity and product formation across many multi-mutants.                                                                                                           |
|                                                                                              | G57L introduces a bulky hydrophobic side chain at a non-pocket glycine site, likely restricting backbone flexibility and potentially destabilizing or altering folding (consistent with very negative 'Unk area' signals) while still allowing activity gains in some contexts.                                                                                |
| S61F*; S61I*                                                                                 | Position 61 is outside the pocket and appears to modulate expression and global dynamics, with effects that can amplify oxygen-transfer activity and product formation when combined with other mutations.                                                                                                                                                     |
|                                                                                              | S61F introduces a bulky aromatic side chain outside the pocket, likely altering local packing and potentially stabilizing a hydrophobic core/surface patch to boost NBD activity and product formation, though it can reduce purified activity in some combinations.                                                                                           |
|                                                                                              | S61I increases hydrophobic packing outside the pocket, likely stabilizing local structure and improving activity moderately, with generally balanced effects on NBD and ABTS.                                                                                                                                                                                  |
| I64L                                                                                         | Position 64 is a hydrophobic pocket-lining residue (~5–8 Å from the ligand) that likely tunes access-channel packing and substrate pose near the reactive center, making it a sensitive lever for shifting peroxygenation/peroxidation balance while modestly impacting expression.                                                                            |
|                                                                                              | I64L is a conservative hydrophobic swap in the pocket that subtly repacks the channel wall near the ligand (~5 Å), likely improving substrate accommodation and boosting both NBD and ABTS activities with minimal stability risk.                                                                                                                             |
| T65K*                                                                                        | Position 65 is outside the defined pocket but is a major epistatic hotspot in multi-mutants, suggesting it influences structural stability/solubility or access-channel dynamics indirectly to tune both activity and selectivity.                                                                                                                             |
|                                                                                              | T65K introduces a positive charge outside the pocket, likely improving solubility/expression and altering long-range electrostatics that enhance overall turnover and product formation across many multi-mutants.                                                                                                                                             |
| T66M*                                                                                        | Position 66 is outside the pocket yet shows a large activity boost in the available data, implying it may affect local secondary-structure packing or stability in a way that increases catalytic competence (evidence limited to one variant).                                                                                                                |
|                                                                                              | T66M replaces a polar hydroxyl with a hydrophobic thioether outside the pocket, likely improving local packing/stability and indirectly enabling a large activity increase (single data point, so uncertain).                                                                                                                                                  |
| T67A*                                                                                        | Position 67 is outside the pocket but repeatedly co-occurs with improved ABTS and NBD in multi-mutants, consistent with an indirect role in shaping access-channel dynamics or overall folding that impacts substrate processing.                                                                                                                              |
|                                                                                              | T67A removes a polar hydroxyl outside the pocket, likely increasing local hydrophobic packing and flexibility in a way that improves substrate access and boosts both NBD and ABTS in many combinations.                                                                                                                                                       |
| M70F                                                                                         | Position 70 sits in the binding pocket close to the ligand (~4 Å min distance) and likely acts as a steric/hydrophobic gate that controls aromatic positioning and residence time, thereby influencing both activity and chemoselectivity.                                                                                                                     |
|                                                                                              | M70F increases aromatic bulk at a near-pocket position (~4.2 Å), likely enhancing hydrophobic/π interactions that stabilize substrate binding and raise both NBD and ABTS activities, though it may reduce channel flexibility.                                                                                                                                |
| S75A*; S75R*                                                                                 | Position 75 is a close pocket residue (~4.8–5.1 Å) likely involved in gating and substrate positioning near the reactive center, making it a strong determinant of oxygen-transfer efficiency and chemoselectivity.                                                                                                                                            |
|                                                                                              | S75A removes a pocket hydroxyl near the reactive center (~5 Å), likely reducing polarity and steric constraints to improve substrate accommodation and generally increase activity while maintaining chemoselectivity.                                                                                                                                         |
|                                                                                              | S75R introduces a bulky positive charge at a close pocket position (~5 Å), likely disrupting substrate binding/orientation and increasing electrostatic barriers, boosting NBD in this context but reducing ABTS, consistent with a strong selectivity shift.                                                                                                  |
| M79L                                                                                         | Position 79 is a pocket residue at intermediate distance (~7–10 Å) that likely shapes the distal part of the access channel and affects substrate ingress/egress and orientation, with moderate effects on activity and selectivity.                                                                                                                           |
|                                                                                              | M79L slightly reduces side-chain polarizability while keeping hydrophobic volume in the pocket, likely smoothing the channel surface to improve oxygen-transfer turnover (higher NBD) with only modest impact on ABTS.                                                                                                                                         |
| T87G                                                                                         | Position 87 is outside the defined pocket and likely affects activity indirectly through local backbone flexibility or folding/trafficking rather than direct substrate binding, consistent with assay shifts that may reflect altered enzyme stability or expression.                                                                                         |
|                                                                                              | T87G removes a side chain outside the pocket, likely increasing local backbone flexibility and subtly altering global dynamics to favor ABTS oxidation and product formation while slightly compromising NBD activity.                                                                                                                                         |
| T88N*                                                                                        | Position 88 is outside the pocket but highly recurrent in improved multi-mutants, suggesting it modulates local structure/dynamics that affect substrate access or peroxide handling indirectly rather than direct binding.                                                                                                                                    |
|                                                                                              | T88N adds a polar amide outside the pocket, likely stabilizing local hydrogen-bond networks and improving folding/solubility, indirectly supporting higher activity and product formation across many multi-mutants.                                                                                                                                           |
| I64L+S182V+Y212K;                                                                            | This cluster combines pocket reshaping (positions 38/64/75/182) with strong surface electrostatic/expression drivers (notably 212 and 236), yielding consistent ABTS gains and moderate NBD gains consistent with improved access/turnover plus altered peroxygenative:peroxidative balance.                                                                   |
| L38M+S182V+Y212K;                                                                            |                                                                                                                                                                                                                                                                                                                                                                |
| T65K+S75A+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                                                                                                |
| T88N+H143T+Y212K+Q236L                                                                       |                                                                                                                                                                                                                                                                                                                                                                |
| I64L+T67A+F220L                                                                              | Combining subtle pocket repacking at 64 with removal of an aromatic gate at 220 and an indirect stabilizing change at 67 likely opens the channel and favors peroxidative ABTS turnover while leaving NBD near baseline due to less precise substrate pre-organization.                                                                                        |
| S41A+S61F+Y212K                                                                              | This combination of non-pocket stability/packing changes (41/61) with a strong surface charge mutation (212) likely boosts overall catalytic competence and product formation primarily via improved folding/expression rather than direct pocket remodeling.                                                                                                  |
| I64L+T67A+H143T;                                                                             | Across these multi-mutants, the shared net effect is channel opening/reshaping at multiple pocket walls (38/64/75/182/220/223) paired with surface electrostatic tuning (212/236 and others), producing strong ABTS increases and moderate NBD gains consistent with enhanced access and a peroxidase-leaning activity profile.                                |
| L38M+T67A;                                                                                   |                                                                                                                                                                                                                                                                                                                                                                |
| L38M+T67A+Y212K;                                                                             |                                                                                                                                                                                                                                                                                                                                                                |
| S217P+F223L;                                                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+T65K;                                                                                   |                                                                                                                                                                                                                                                                                                                                                                |
| T65K+H143T+Y212K;                                                                            |                                                                                                                                                                                                                                                                                                                                                                |
| T65K+S75A+T88N+Y212K;                                                                        |                                                                                                                                                                                                                                                                                                                                                                |
| T65K+T88N+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                                                                                                |
| T65K+Y212K+Q236L;                                                                            |                                                                                                                                                                                                                                                                                                                                                                |
| T67A+S182V+Y212K;                                                                            |                                                                                                                                                                                                                                                                                                                                                                |
| Y212K+F220L                                                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| T88N+Y212K+Q236L                                                                             | T88N+Y212K+Q236L likely improves folding/solubility and long-range electrostatics without direct pocket changes, giving modest parallel increases in NBD and ABTS consistent with a global activity uplift.                                                                                                                                                    |
| H143T+Y212K;                                                                                 | This cluster is dominated by surface/stability mutations (29/61/65/88/143/212/214/236) with limited pocket edits (38/64/75/182), yielding moderate gains in both assays consistent with improved expression and dynamics plus mild channel tuning.                                                                                                             |
| I64L+S182V;                                                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| L38M+I64L+Y212K;                                                                             |                                                                                                                                                                                                                                                                                                                                                                |
| L38M+Y212K;                                                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| S182V+Y212K;                                                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| S29A+G57A+S214P;                                                                             |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+G57L;                                                                                   |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+T65K+V138T;                                                                             |                                                                                                                                                                                                                                                                                                                                                                |
| S61I+Y212K;                                                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| S75A+H143T+Y212K+Q236L;                                                                      |                                                                                                                                                                                                                                                                                                                                                                |
| T65K+T88N+Y212K                                                                              |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+F223L;                                                                                  | These variants largely combine surface/stability tuning (29/41/61/65/88/143/212/236) with a pocket gate change at 223, resulting in modest NBD gains but near-neutral ABTS on average, consistent with shifting toward oxygen-transfer/product formation rather than strong peroxidase enhancement.                                                            |
| S41A+S75A+T88N+Y212K;                                                                        |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+S75A+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+T65K+H143T+Y212K;                                                                       |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+T65K+S75A+H143T;                                                                        |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+T65K+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+T88N+Y212K;                                                                             |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+Y212K+Q236L;                                                                            |                                                                                                                                                                                                                                                                                                                                                                |
| S61I+T65K+S75A+Y212K;                                                                        |                                                                                                                                                                                                                                                                                                                                                                |
| S61I+T88N+H143T+Y212K;                                                                       |                                                                                                                                                                                                                                                                                                                                                                |
| T65K+Q236L                                                                                   |                                                                                                                                                                                                                                                                                                                                                                |
| H143Q+F220L;                                                                                 | Pairing a non-pocket histidine change at 143 with removal of an aromatic pocket feature at 220 likely disrupts productive oxygen-transfer binding (lower NBD) while maintaining elevated ABTS via easier access and altered electrostatics.                                                                                                                    |
| H143T+F220L                                                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| L38M+I64L+S182V;                                                                             | These combinations mix pocket reshaping (38/64/70/182/220) with non-pocket stability/electrostatic changes (61/67/88/212/236/217), yielding strong ABTS improvements but near-baseline NBD, consistent with enhanced peroxidative turnover and access rather than optimized oxygen-transfer pose.                                                              |
| L38M+T67A+S182V;                                                                             |                                                                                                                                                                                                                                                                                                                                                                |
| M70F+S217P+F220L;                                                                            |                                                                                                                                                                                                                                                                                                                                                                |
| S61I+T88N+Y212K+Q236L                                                                        |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+T65K+Y212K;                                                                             | Adding strong surface electrostatic changes (65/212 with or without 143/75/41) yields improved product formation but suppresses ABTS in this subset, suggesting these combinations bias away from peroxidative electron-transfer pathways while only modestly improving oxygen transfer.                                                                       |
| T65K+S75A+H143T+Y212K                                                                        |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+I64L+T65K+A171F+S182M+Y212K;                                                            | This high-performing cluster combines major surface/expression drivers (29/65/88/143/212/236) with distal pocket remodeling (171/182 and 64/75), producing large NBD gains and improved purified activity consistent with synergistic channel reshaping plus improved folding/solubility.                                                                      |
| T65K+S75A+H143T+Y212K+Q236L;                                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| T67A+T88N+H143T+A171F+S182M+Y212K                                                            |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+T67A+T88N+H143T+A171F+S182M+H208D                                                       | Stacking multiple surface/dynamic mutations with distal pocket remodeling (171/182) and an electrostatic change at 208 yields strong NBD and purified activity with only modest ABTS, consistent with favoring peroxygenation over peroxidation.                                                                                                               |
| S41A+G57A+T65K+A171F+S182M+Q236L                                                             | Despite strong NBD gains from combined stability and pocket remodeling, the extremely low protein yield suggests this combination destabilizes expression/folding, so apparent activity improvements may be offset by poor producibility.                                                                                                                      |
| S29P+S41A+T67A+T88N+A171F+Q236L                                                              | This set of mostly non-pocket stabilizing mutations plus distal pocket remodeling at 171 likely boosts NBD substantially while keeping ABTS low, consistent with improved oxygen-transfer competence without enhancing peroxidative pathways.                                                                                                                  |
| T65K+S75A+T88N+A171F+Y212K+H208D+Q236L;                                                      | Combining surface electrostatic/stability changes (65/88/212/236 with 143 and/or 208) with pocket gating at 75 and distal pocket remodeling at 171 yields strong NBD and good ABTS, consistent with both improved access and improved catalytic competence, though side-reaction signals suggest possible instability or assay interference in one background. |
| T65K+S75A+T88N+H143T+Y212K                                                                   |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+T65K+T88N+Y212K+Q236L                                                                   | This surface-focused combination (41/65/88/212/236) likely improves folding/solubility and long-range electrostatics, giving a sizeable NBD increase with modest ABTS consistent with a global activity uplift rather than direct pocket optimization.                                                                                                         |
| S41A+T65K+H143T+Y212K+Q236L                                                                  | Adding the 143 change to the 41/65/212/236 background modestly improves NBD but depresses purified ABTS, consistent with electrostatic rewiring that favors oxygen transfer while reducing peroxidative turnover.                                                                                                                                              |
| S29P+S41A+T65K+S75A+T88N+A171F+Y212K+S182M;                                                  | When distal pocket remodeling (171/182) is combined with multiple surface/stability mutations and pocket gating at 75, NBD can become very high while ABTS remains moderate, consistent with synergistic channel reshaping that favors productive oxygen-transfer binding and improved expression.                                                             |
| S41A+T65K+T88N+H143T+Y212K                                                                   |                                                                                                                                                                                                                                                                                                                                                                |
| G57A+I64L+T65K+H143T+A171F+H208D;                                                            | This broad cluster repeatedly combines expression/stability boosters (29/41/57/65/88/143/212/236) with distal pocket remodeling (171/182) and occasional electrostatic tuning (208), yielding consistently high NBD and moderate ABTS consistent with enhanced peroxygenation capacity plus improved enzyme availability.                                      |
| G57A+I64L+T65K+T88N+H143T+A171F+H208D+Y212K;                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| G57A+I64L+T88N+H143T+A171F+S182M+Y212K;                                                      |                                                                                                                                                                                                                                                                                                                                                                |
| G57A+T65K+T67A+H143T+A171F+H208D+Q236L;                                                      |                                                                                                                                                                                                                                                                                                                                                                |
| G57A+T65K+T67A+T88N+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| I64L+T65K+T67A+T88N+A171F+S182M+Y212K;                                                       |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+G57A+I64L+T65K+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+G57A+I64L+T88N+H143T+A171F+S182M+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+G57A+T67A+T88N+H143T+A171F+S182M+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+G57A+I64L+T67A+A171F+H208D+Y212K;                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+G57A+T67A+T88N+A171F+S182M+H208D;                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+T65K+T67A+T88N+H143T+A171F+Y212K;                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+T65K+T67A+T88N+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+I64L+T65K+T88N+H143T+A171F+S182M+H208D;                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+I64L+T65K+T88N+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+I64L+T88N+A171F+Y212K;                                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| S61I+T65K+H143T+Y212K+Q236L;                                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| S75A+T88N+H143T+Y212K+Q236L;                                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| T65K+S75A+T88N+A171F+Y212K+Q236L;                                                            |                                                                                                                                                                                                                                                                                                                                                                |
| T65K+S75A+T88N+Y212K+Q236L;                                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| T65K+T67A+T88N+H143T+A171F+H208D+Y212K;                                                      |                                                                                                                                                                                                                                                                                                                                                                |
| T65K+T67A+T88N+H143T+A171F+Q236L                                                             |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+G57A+H143T+A171F+S182M+H208D                                                            | This combination of surface/dynamic mutations with distal pocket remodeling and 208 electrostatic tuning increases NBD but suppresses ABTS, consistent with shifting the enzyme away from peroxidative electron-transfer chemistry.                                                                                                                            |
| S29P+G57A+T88N+H143T+A171F+H208D+Y212K+Q236L;                                                | These multi-mutants combine extensive surface/stability tuning with distal pocket remodeling (171/182) and limited pocket edits (64/75), producing strong NBD but generally low ABTS, consistent with favoring oxygen transfer while dampening peroxidative activity.                                                                                          |
| S29P+S41A+G57A+I64L+A171F+Y212K;                                                             |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+G57A+T67A+H143T+A171F+Y212K;                                                       |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+I64L+A171F+Y212K;                                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+I64L+T88N+A171F+S182M+H208D+Q236L;                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+T67A+T88N+H143T+A171F+S182M+H208D;                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+T88N+H143T+A171F+S182M+Y212K;                                                           |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+G57A+I64L+T67A+T88N+H143T+A171F+Q236L;                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+G57A+I64L+T88N+H143T+A171F+H208D;                                                       |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+G57A+T67A+H143T+A171F+Y212K;                                                            |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+T65K+S75A+H143T+Y212K;                                                                  |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+T65K+T88N+H143T+Y212K+Q236L;                                                            |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+T67A+H143T+A171F+Q236L;                                                                 |                                                                                                                                                                                                                                                                                                                                                                |
| T65K+S75A+T88N+H143T+A171F+Y212K+Q236L                                                       |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+S75A+H143T+Y212K+Q236L                                                                  | Adding pocket gating at 75 to a surface-electrostatic background yields near-baseline activities, suggesting negative epistasis where channel changes and electrostatic tuning partially cancel benefits (uncertain with single data point).                                                                                                                   |
| S41A+G57A+I64L+T65K+T67A+T88N+A171F+S182M+H208D+Q236L                                        | This heavily stacked variant achieves very high NBD but extremely low yield, consistent with strong catalytic enhancement from combined stability and distal pocket remodeling that is offset by poor expression/folding in this background.                                                                                                                   |
| S29P+S41A+G57A+H143T+A167E+A171F+S182M+H208D+Q236L                                           | Introducing an additional negative charge mutation (167) into an already stacked background boosts NBD but collapses ABTS, consistent with electrostatic tuning that disfavors peroxidative pathways while maintaining oxygen-transfer competence.                                                                                                             |
| S29P+S41A+T65K+T67A+T88N+A171F+H208D+Y212K+Q236L                                             | This stacked surface/electrostatic and distal pocket remodeling combination yields high NBD but low ABTS and low yield, consistent with strong peroxygenation bias coupled to expression/stability penalties.                                                                                                                                                  |
| S29P+S41A+G57A+T65K+T88N+H143T+A171F+S182M+Y212K+Q236L                                       | This combination yields strong NBD with near-neutral ABTS and good yield, consistent with a peroxygenation-favoring balance achieved by coupling surface stability mutations with distal pocket remodeling and electrostatic tuning.                                                                                                                           |
| S29P+S41A+G57A+I64L+T65K+T67A+H143T+A171F+S182M+H208D+Y212K+Q236L                            | Despite very high NBD and ABTS, the very low purified activities alongside extremely high yield suggest substantial assay-context differences (e.g., secretion/lysate effects or inactive protein fraction), so the mechanistic interpretation is uncertain.                                                                                                   |
| S29P+S41A+I64L+T65K+S75A+T88N+H143T+A171F+S182M+Y212K+Q236L                                  | This variant shows the strongest NBD gain with elevated ABTS, consistent with synergistic pocket gating (64/75) plus distal pocket remodeling (171/182) and surface stability/electrostatic tuning (29/41/65/88/143/212/236) that jointly optimize access and productive binding.                                                                              |
| G57A+I64L+T67A+T88N+H143T+A171F+S182M+Y212K+Q236L;                                           | Across these highly stacked mutants, the shared net effect is strong global stabilization/expression tuning plus distal pocket remodeling that drives high NBD and moderate ABTS, while additional C-terminal charge/packing edits likely further modulate solubility and trafficking rather than direct pocket chemistry.                                     |
| I64L+T67A+H143T+A171F+H208D+Y212K+Q236L+S237V+R239E+A240Q+I241S+E242S+L243C;                 |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+G57A+I64L+T65K+T67A+H143T+A171F+S182M+Y212K;                                            |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+G57A+I64L+T65K+T67A+T88N+H143T+A171F+Y212K+Q236L;                                       |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+G57A+I64L+T65K+H143T+A171F+S182M+H208D;                                            |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+G57A+I64L+T65K+T66M+T67A+A171F+S182M+Y212K+Q236L;                                  |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+G57A+I64L+T65K+T67A+A171F+S182M+H208D+Q236L;                                       |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+G57A+I64L+T88N+A171F+S182M+Y212K+Q236L;                                            |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+G57A+I64L+T88N+H143T+A171F+S182M+Y212K;                                            |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+G57A+I64L+T88N+H143T+A171F+S182M+Y212K+Q236L;                                      |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+G57L+S61F+I64L+T65K+T67A+S75A+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L;            |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+T65K+T67A+T88N+H143T+A171F+S182M+E197K+H208D+Y212K+Q236L;                               |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+I64L+T65K+S75A+T88N+H143T+A171F+H208D+Y212K+Q236L;                                      |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+I64L+T65K+S75A+T88N+H143T+A171F+Y212K+Q236L                                             |                                                                                                                                                                                                                                                                                                                                                                |
| S29P+S41A+S61F+I64L+T65K+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L                            | Adding a bulky hydrophobic mutation at 61 into a heavily stacked background yields moderate NBD but low ABTS, consistent with altered folding/packing that maintains oxygen-transfer competence while reducing peroxidative turnover.                                                                                                                          |
| S29P+S41A+G57L+I64L+T65K+S75A+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L;                      | These variants combine extensive surface/stability tuning with distal pocket remodeling and additional C-terminal electrostatic/packing edits, yielding moderate NBD and low ABTS consistent with improved enzyme availability but a peroxygenation-biased, peroxidase-suppressed profile.                                                                     |
| S41A+G57A+I64L+T67A+H143T+A171F+S182M+H208D+Y212K+Q236L+S237V+R239E+A240Q+I241S+E242S+L243C; |                                                                                                                                                                                                                                                                                                                                                                |
| S41A+I64L+T65K+S75R+T88N+H143T+A171F+Y212K+Q236L                                             |                                                                                                                                                                                                                                                                                                                                                                |

{'thread_id': '0bdc9892f0ee4ad1a6e91ee71e23f30f',
 'step_processed_dir': '/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/15_analyze_mutants',
 'explanations_csv_path': '/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/15_analyze_mutants/mutant_effect_explanations.csv',
 'llm_analysis_path': '/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/15_analyze_mutants/mutant_analysis_llm_summary.md'}

## Reflect And Improve Output

Optionally provide critique or additional instructions, then ask the LLM to revise the explanation table and overwrite the saved CSV/markdown outputs.


In [5]:
reflection_user_feedback = {
    "mutant_analysis_reflection_user_feedback": "Reduce redundancy in descriptions. For mutations to Met near the binding pocket heme, consider that these could improve tolerance to hydrogen peroxide and thus lifetime of the enzyme. Reduce explicit mentions of NBD and ABTS activity; instead refer more generally to peroxygenation and peroxidation activity. Finally, simplify the language a bit, suitable for a technically-trained generalist.",  # Optional: additional critique or revision instructions
    "mutant_analysis_reflection_prompt_override": "",  # Optional: replace the default reflection prompt
}

reflection_feedback = str(reflection_user_feedback.get("mutant_analysis_reflection_user_feedback", "")).strip()
reflection_prompt = str(reflection_user_feedback.get("mutant_analysis_reflection_prompt_override", "")).strip() or MUTANT_ANALYSIS_REFLECTION_PROMPT

reflection_outputs = reflect_and_regenerate_mutant_explanations(
    analysis_units_df=result["analysis_units_df"],
    current_explanations_df=result["explanations_df"],
    mutant_df=result["mutant_df"],
    binding_summary_row=result["binding_summary_row"],
    user_inputs=user_inputs,
    supplemental_context=(
        str(result["binding_pocket_context_result"].get("filtered_context_text") or result["binding_pocket_context_result"].get("context_text") or "").strip()
        + ("\n\n" if str(result["binding_pocket_context_result"].get("filtered_context_text") or result["binding_pocket_context_result"].get("context_text") or "").strip() and str(result["literature_context_result"].get("context_text", "")).strip() else "")
        + str(result["literature_context_result"].get("context_text", "")).strip()
    ),
    user_feedback=reflection_feedback,
    critique_prompt=reflection_prompt,
    original_prompt_text=result.get("prompt_text", ""),
    original_unit_level_output_json=result.get("llm_json_text", ""),
)

result["explanations_df"] = reflection_outputs["explanations_df"]
result["llm_json_text"] = reflection_outputs["llm_json_text"]

out_explanations_csv = save_mutant_explanations_csv(result["explanations_df"], result["step_processed_dir"], filename="mutant_effect_explanations_revised.csv")
out_llm = save_llm_analysis(
    "Mutant analysis reflection prompt:\n\n"
    + reflection_outputs["prompt_text"]
    + "\n\nRefined explanation table:\n\n"
    + result["explanations_df"].to_markdown(index=False)
    + "\n\nRaw LLM JSON:\n```json\n"
    + result["llm_json_text"]
    + "\n```",
    result["step_processed_dir"],
)

result["explanations_csv_path"] = out_explanations_csv
result["llm_analysis_path"] = out_llm

{
    "refined_csv_path": str(out_explanations_csv),
    "refined_llm_summary_path": str(out_llm),
}


Critique and revisions summary:
- Reduced repetitive phrasing by collapsing “position summary + mutant-specific restatement” into shorter, non-duplicative rationales.  
- Shifted language away from assay-specific readouts (NBD/ABTS) toward mechanism-level terms: peroxygenation vs peroxidation and overall turnover/selectivity.  
- Simplified and generalized mechanistic claims (less over-precise causal wording), making entries more readable for a technically trained generalist.  
- Clarified pocket vs non-pocket logic: pocket mutations framed as channel/gating/pre-organization effects; surface mutations framed as stability/solubility/expression and long-range electrostatics.  
- Added/strengthened the interpretation that Met substitutions near the heme/pocket may improve H₂O₂ tolerance/oxidative robustness (enzyme lifetime), not just binding geometry.  
- Increased explicit uncertainty/context-dependence where evidence is sparse or epistatic (e.g., limited single-mutant data, background-

### Mutant Analysis Reflection / Rewrite

<details><summary>Prompt</summary>

```text
You are reviewing an existing mutant-effect explanation table for a protein engineering workflow.

Task:
Improve the current explanations using the original analysis context plus user-supplied critique.

Output contract (strict):
- Return ONLY a JSON array.
- Return one object per provided analysis unit.
- Each object must contain:
  - row_index: integer copied from the provided analysis unit row_index
  - Description of effect: one sentence, revised and improved
- Do not return markdown, code fences, or extra prose.

Rules:
- Preserve coverage of all provided rows.
- Keep each explanation concise, specific, and technically grounded.
- Incorporate user feedback where compatible with the provided context.
- Do not invent unsupported mechanistic claims.
- For single-position rows, keep the explanation position-centric: explain why the residue position matters, and do not describe a specific amino-acid substitution.
- For single-substitution rows, focus on the effect of the specific substitution itself.
- For a position row and a substitution row at the same residue, the two explanations must be meaningfully different and should not repeat the same sentence in paraphrased form.
```
</details>

#### Response

(Refined explanation table shown below.)

### Refined Mutant Explanation Table

| Mutant(s)                                                                                    | Description of effect                                                                                                                                                                                                                                                                      |
|:---------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| V138T*                                                                                       | Position 138 is outside the pocket, so observed effects are most consistent with indirect changes to stability or long-range packing rather than altered substrate binding.                                                                                                                |
|                                                                                              | V138T introduces a polar hydroxyl outside the pocket, consistent with modest stabilization via improved local hydrogen bonding/hydration and small activity shifts.                                                                                                                        |
| H143Q*; H143T*                                                                               | Position 143 is outside the pocket and appears to tune long-range electrostatics/protonation networks, showing strong context dependence across multi-mutants.                                                                                                                             |
|                                                                                              | H143Q removes histidine protonation capability at a surface site, consistent with altered electrostatics that can reduce peroxygenation while relatively favoring peroxidation in some contexts.                                                                                           |
|                                                                                              | H143T replaces histidine with a neutral polar residue at a surface site, consistent with a gentler electrostatic change that can improve overall performance in many multi-mutant backgrounds.                                                                                             |
| A167E*                                                                                       | Position 167 is outside the pocket and likely influences surface electrostatics or stability, but evidence is limited to a small number of observations.                                                                                                                                   |
|                                                                                              | A167E introduces a negative charge at a non-pocket site, consistent with altered surface electrostatics that shifts the peroxygenation/peroxidation balance (and may affect expression) in a context-dependent way.                                                                        |
| A171F; A171I; A171L; A171V                                                                   | Position 171 is a distal pocket-lining site (~6.8–10.6 Å) where side-chain size and hydrophobicity can remodel the channel wall and shift substrate positioning without directly contacting the reactive center.                                                                           |
|                                                                                              | A171F replaces a small side chain with a bulky aromatic group in the distal pocket, likely narrowing/reshaping the channel to favor some binding poses while disfavoring others, consistent with a strong selectivity shift.                                                               |
|                                                                                              | A171I increases hydrophobic bulk at a distal pocket wall, consistent with improved substrate packing and residence time without introducing new polarity or charge.                                                                                                                        |
|                                                                                              | A171L adds hydrophobic volume at the distal pocket boundary, likely tightening the channel and biasing substrate orientation in a way that can trade peroxygenation against peroxidation.                                                                                                  |
|                                                                                              | A171V modestly increases hydrophobic packing at the distal pocket wall, consistent with a milder channel reshaping that can improve activity without severe steric penalties.                                                                                                              |
| L174F                                                                                        | Position 174 is a very close pocket contact (~3.4 Å) that likely serves as a tight steric gate, so small geometric changes here can strongly alter access and productive binding.                                                                                                          |
|                                                                                              | L174F introduces a bulky aromatic side chain at a very tight pocket contact (~3.4 Å), likely creating steric crowding that impairs some productive binding modes while favoring others.                                                                                                    |
| S182A; S182C; S182L; S182M; S182V                                                            | Position 182 is a distal pocket residue (~7 Å) that likely tunes local polarity/packing near a secondary cavity or channel exit, influencing binding dynamics and the peroxygenation/peroxidation tradeoff.                                                                                |
|                                                                                              | S182A removes a distal-pocket hydroxyl, likely reducing local polarity and hydrogen bonding to subtly favor hydrophobic binding and alter selectivity.                                                                                                                                     |
|                                                                                              | S182C replaces a hydroxyl with a thiol, changing polarity and polarizability at the distal pocket wall in a way that can modestly reshape binding and reactivity.                                                                                                                          |
|                                                                                              | S182L introduces a larger hydrophobic side chain at the distal pocket, likely tightening the channel and increasing substrate residence time.                                                                                                                                              |
|                                                                                              | S182M introduces a thioether in the pocket, which can both adjust hydrophobic packing and (in UPO contexts) plausibly improve oxidative robustness, supporting higher sustained activity.                                                                                                  |
|                                                                                              | S182V increases hydrophobicity with a branched side chain at the distal pocket, consistent with a binding-pose shift that can trade peroxygenation efficiency against peroxidation.                                                                                                        |
| E197K*                                                                                       | Position 197 is outside the pocket and likely affects folding/solubility via surface charge networks, indirectly shifting overall activity.                                                                                                                                                |
|                                                                                              | E197K reverses surface charge outside the pocket, consistent with improved solubility/production and a global activity increase rather than a binding-site effect.                                                                                                                         |
| H208D                                                                                        | Position 208 lies outside the pocket, so its effects most plausibly arise from long-range electrostatics or stability changes that alter overall catalytic competence rather than substrate binding.                                                                                       |
|                                                                                              | H208D replaces a titratable side chain with a fixed negative charge on the surface, plausibly rewiring local electrostatics and shifting the peroxygenation/peroxidation balance indirectly.                                                                                               |
| Y212K; Y212T                                                                                 | Position 212 is outside the pocket yet repeatedly associates with higher activity and yield, consistent with a surface hotspot that modulates expression/solubility or long-range electrostatics impacting overall turnover.                                                               |
|                                                                                              | Y212K introduces a strong positive charge at a surface position, consistent with improved expression/solubility and altered long-range electrostatics, though the extreme 'Unk area' signal suggests possible assay interference or side chemistry.                                        |
|                                                                                              | Y212T removes an aromatic ring and reduces side-chain size at a surface position, consistent with improved folding/production and a moderate global activity uplift.                                                                                                                       |
| S214P*                                                                                       | Position 214 is outside the pocket and likely sits in a loop/turn where backbone rigidity can influence folding, stability, or trafficking.                                                                                                                                                |
|                                                                                              | S214P introduces proline-mediated rigidity outside the pocket, consistent with stabilizing a loop/turn and improving folding or trafficking.                                                                                                                                               |
| S217P*                                                                                       | Position 217 is outside the pocket and likely affects loop dynamics that can couple to access-channel motions or overall stability, with strong effects in some combinations.                                                                                                              |
|                                                                                              | S217P introduces a rigid proline outside the pocket, consistent with altered loop dynamics that can change access-channel motions and shift selectivity.                                                                                                                                   |
| F220L                                                                                        | Position 220 is a pocket residue (~6.4 Å) that contributes to hydrophobic/aromatic packing in the channel, so perturbations here can strongly shift substrate pre-organization and selectivity.                                                                                            |
|                                                                                              | F220L removes an aromatic ring from a pocket-lining position, likely weakening π/hydrophobic pre-organization and loosening packing, which can reduce productive binding while allowing faster, less selective turnover.                                                                   |
| F223L*                                                                                       | Position 223 is a close pocket contact (~3.8 Å) that likely forms part of a hydrophobic gate controlling substrate approach and residence time.                                                                                                                                            |
|                                                                                              | F223L removes an aromatic ring at a close pocket gate, likely enlarging/softening the channel to increase throughput while reducing precise substrate pre-organization.                                                                                                                    |
| Q236L*                                                                                       | Position 236 is outside the pocket but highly recurrent in multi-mutants, consistent with a stability/solubility lever that modulates overall catalytic output rather than direct binding.                                                                                                 |
|                                                                                              | Q236L removes a polar amide outside the pocket, consistent with increased hydrophobic packing and stability that broadly supports higher activity across combinations.                                                                                                                     |
| S237V*                                                                                       | Position 237 is outside the pocket near the C-terminus, so effects are most consistent with local packing/flexibility changes that indirectly influence activity.                                                                                                                          |
|                                                                                              | S237V increases hydrophobicity outside the pocket, consistent with improved local packing and stability near the C-terminus.                                                                                                                                                               |
| R239E*                                                                                       | Position 239 is outside the pocket and likely affects surface electrostatics and salt-bridge patterns, with effects that are strongly background-dependent.                                                                                                                                |
|                                                                                              | R239E reverses charge at a surface position, likely rewiring salt-bridge networks and solubility with strongly background-dependent effects (no isolated single-mutant readout here).                                                                                                      |
| A240Q*                                                                                       | Position 240 is outside the pocket and likely tunes local packing/polarity on the surface, contributing indirectly to stability or expression in multi-mutants.                                                                                                                            |
|                                                                                              | A240Q adds a polar amide at a non-pocket position, consistent with increased local hydrogen bonding and solubility that can support higher activity in multi-mutant backgrounds.                                                                                                           |
| I241S*                                                                                       | Position 241 is outside the pocket and likely sits in a region where hydrophobic-to-polar balance affects local stability and solubility.                                                                                                                                                  |
|                                                                                              | I241S introduces a polar hydroxyl at a non-pocket hydrophobic position, consistent with increased local hydration/solubility and altered packing near the C-terminus.                                                                                                                      |
| E242S*                                                                                       | Position 242 is outside the pocket and likely influences surface charge/polarity, indirectly affecting folding and functional expression.                                                                                                                                                  |
|                                                                                              | E242S removes a negative charge outside the pocket, consistent with reduced electrostatic frustration and improved folding/solubility that indirectly increases activity.                                                                                                                  |
| L243C*                                                                                       | Position 243 is outside the pocket near the C-terminus, so effects are most consistent with subtle local packing changes and remain uncertain without single-mutant data.                                                                                                                  |
|                                                                                              | L243C introduces a smaller, more polarizable side chain outside the pocket, which may subtly alter local packing or redox sensitivity, but the mechanism is uncertain.                                                                                                                     |
| S29A*; S29P*                                                                                 | Position 29 is outside the pocket but repeatedly appears in high-performing combinations, consistent with a structural/trafficking hotspot that modulates enzyme availability rather than active-site chemistry.                                                                           |
|                                                                                              | S29A removes a polar hydroxyl outside the pocket, consistent with modestly improved local packing and stability that can raise apparent activity.                                                                                                                                          |
|                                                                                              | S29P introduces backbone rigidity outside the pocket, consistent with stabilizing a structural element important for folding/trafficking and enabling large gains in multi-mutant contexts.                                                                                                |
| L38M*                                                                                        | Position 38 is a pocket residue at moderate distance (~7–10 Å) that likely shapes the access channel wall, strongly influencing how readily substrates enter and rebind.                                                                                                                   |
|                                                                                              | L38M introduces a thioether at a pocket-adjacent channel wall, which can adjust packing and (in UPO contexts) plausibly improve oxidative robustness, supporting higher sustained turnover.                                                                                                |
| S41A*                                                                                        | Position 41 is outside the pocket and is enriched among improved variants, consistent with an indirect role in folding stability, secretion, or global dynamics that raises effective enzyme concentration.                                                                                |
|                                                                                              | S41A removes a polar hydroxyl outside the pocket, consistent with improved local packing and reduced misfolding/aggregation that increases effective enzyme levels.                                                                                                                        |
| G57A*; G57L*                                                                                 | Position 57 is outside the pocket and involves a glycine site, so mutations here likely act through backbone conformational control that impacts folding and functional expression.                                                                                                        |
|                                                                                              | G57A replaces glycine with a small side chain outside the pocket, consistent with reduced backbone flexibility and improved folding/trafficking that raises apparent activity across backgrounds.                                                                                          |
|                                                                                              | G57L introduces a bulky hydrophobic side chain at a non-pocket glycine site, likely restricting backbone conformations and causing strong context dependence, including potential folding penalties.                                                                                       |
| S61F*; S61I*                                                                                 | Position 61 is outside the pocket and appears to modulate global packing or surface properties, with effects that amplify performance mainly through enzyme stability/expression rather than binding-site geometry.                                                                        |
|                                                                                              | S61F introduces a bulky aromatic side chain outside the pocket, consistent with altered packing that can boost activity but also create context-dependent stability costs.                                                                                                                 |
|                                                                                              | S61I increases hydrophobic packing outside the pocket, consistent with modest stabilization and broadly balanced effects on activity and selectivity.                                                                                                                                      |
| I64L                                                                                         | Position 64 lines the binding pocket (~5–8 Å from ligand) and is poised to subtly tune channel packing and substrate approach geometry, making it a sensitive determinant of peroxygenation vs peroxidation balance.                                                                       |
|                                                                                              | I64L is a conservative hydrophobic swap in the pocket that subtly repacks the channel wall near the ligand, consistent with modestly improved substrate accommodation and overall activity.                                                                                                |
| T65K*                                                                                        | Position 65 is outside the pocket but highly epistatic across multi-mutants, consistent with a surface electrostatics/solubility lever that broadly tunes activity and selectivity.                                                                                                        |
|                                                                                              | T65K introduces a positive charge outside the pocket, consistent with improved solubility/production and long-range electrostatic effects that broadly increase activity in many combinations.                                                                                             |
| T66M*                                                                                        | Position 66 is outside the pocket and shows a large effect in limited data, suggesting a local packing/stability hotspot but with high uncertainty from sparse sampling.                                                                                                                   |
|                                                                                              | T66M replaces a polar side chain with a thioether outside the pocket, consistent with improved local packing and possibly oxidative robustness, but evidence is limited to one observation.                                                                                                |
| T67A*                                                                                        | Position 67 is outside the pocket yet frequently co-occurs with improved variants, consistent with an indirect role in local structure/dynamics that influences access-channel behavior.                                                                                                   |
|                                                                                              | T67A removes a polar hydroxyl outside the pocket, consistent with increased local hydrophobic packing that indirectly improves access-channel behavior across many combinations.                                                                                                           |
| M70F                                                                                         | Position 70 is a near-heme pocket contact (~4.2 Å) that likely acts as a steric/hydrophobic gate controlling how aromatics sit over the reactive center, strongly influencing turnover and selectivity.                                                                                    |
|                                                                                              | M70F increases aromatic bulk at a near-heme pocket position, consistent with stronger hydrophobic/π interactions that can stabilize binding but also restrict channel flexibility.                                                                                                         |
| S75A*; S75R*                                                                                 | Position 75 is a close pocket residue (~4.8–5.1 Å) near the reactive center that likely gates substrate approach and orientation, making it a strong determinant of selectivity.                                                                                                           |
|                                                                                              | S75A removes a pocket hydroxyl near the reactive center, likely reducing polarity/steric constraints to improve substrate accommodation and shift selectivity.                                                                                                                             |
|                                                                                              | S75R introduces a bulky positive charge at a close pocket position, likely disrupting binding geometry and strongly shifting the peroxygenation/peroxidation balance.                                                                                                                      |
| M79L                                                                                         | Position 79 sits in the pocket but farther from the reactive center (~6.8–9.6 Å), so it likely shapes the distal channel segment that governs substrate ingress/egress and binding pose stability.                                                                                         |
|                                                                                              | M79L removes sulfur polarizability while keeping similar hydrophobic volume in the pocket, consistent with smoother channel packing and a modest shift toward more efficient peroxygenation.                                                                                               |
| T87G                                                                                         | Position 87 is outside the defined pocket, so its effects are most consistent with indirect changes to local backbone flexibility or folding/trafficking rather than direct substrate contacts.                                                                                            |
|                                                                                              | T87G removes a side chain outside the pocket, likely increasing local flexibility and indirectly shifting stability or dynamics that affect apparent activity and product formation.                                                                                                       |
| T88N*                                                                                        | Position 88 is outside the pocket but recurrent in improved variants, consistent with a stabilizing or folding-related role that indirectly increases effective catalytic turnover.                                                                                                        |
|                                                                                              | T88N adds a polar amide outside the pocket, consistent with stabilizing local hydrogen-bonding networks and improving folding/solubility in multi-mutant backgrounds.                                                                                                                      |
| I64L+S182V+Y212K;                                                                            | This cluster combines pocket/channel reshaping (38/64/75/182) with surface stability/electrostatic drivers (notably 212/236), yielding a consistent global activity uplift with a tendency toward higher peroxidation.                                                                     |
| L38M+S182V+Y212K;                                                                            |                                                                                                                                                                                                                                                                                            |
| T65K+S75A+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                            |
| T88N+H143T+Y212K+Q236L                                                                       |                                                                                                                                                                                                                                                                                            |
| I64L+T67A+F220L                                                                              | I64L+T67A+F220L likely opens and loosens the pocket (loss of aromatic packing at 220 plus subtle channel repacking), favoring faster but less pre-organized turnover and shifting selectivity toward peroxidation.                                                                         |
| S41A+S61F+Y212K                                                                              | S41A+S61F+Y212K combines surface packing and charge changes that most plausibly improve folding/production and overall catalytic competence rather than directly remodeling the active site.                                                                                               |
| I64L+T67A+H143T;                                                                             | Across these multi-mutants, the shared pattern is channel opening/reshaping at several pocket walls (38/64/75/182/220/223) plus surface electrostatic tuning (often 212/236), producing strong peroxidation gains with moderate peroxygenation changes.                                    |
| L38M+T67A;                                                                                   |                                                                                                                                                                                                                                                                                            |
| L38M+T67A+Y212K;                                                                             |                                                                                                                                                                                                                                                                                            |
| S217P+F223L;                                                                                 |                                                                                                                                                                                                                                                                                            |
| S29P+T65K;                                                                                   |                                                                                                                                                                                                                                                                                            |
| T65K+H143T+Y212K;                                                                            |                                                                                                                                                                                                                                                                                            |
| T65K+S75A+T88N+Y212K;                                                                        |                                                                                                                                                                                                                                                                                            |
| T65K+T88N+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                            |
| T65K+Y212K+Q236L;                                                                            |                                                                                                                                                                                                                                                                                            |
| T67A+S182V+Y212K;                                                                            |                                                                                                                                                                                                                                                                                            |
| Y212K+F220L                                                                                  |                                                                                                                                                                                                                                                                                            |
| T88N+Y212K+Q236L                                                                             | T88N+Y212K+Q236L is dominated by surface stability/charge effects with no direct pocket edits, consistent with a modest global activity increase rather than a selectivity redesign.                                                                                                       |
| H143T+Y212K;                                                                                 | This cluster is dominated by surface/folding mutations (29/61/65/88/143/212/214/236) with limited pocket edits, consistent with improved enzyme availability plus mild channel tuning.                                                                                                     |
| I64L+S182V;                                                                                  |                                                                                                                                                                                                                                                                                            |
| L38M+I64L+Y212K;                                                                             |                                                                                                                                                                                                                                                                                            |
| L38M+Y212K;                                                                                  |                                                                                                                                                                                                                                                                                            |
| S182V+Y212K;                                                                                 |                                                                                                                                                                                                                                                                                            |
| S29A+G57A+S214P;                                                                             |                                                                                                                                                                                                                                                                                            |
| S29P+G57L;                                                                                   |                                                                                                                                                                                                                                                                                            |
| S29P+T65K+V138T;                                                                             |                                                                                                                                                                                                                                                                                            |
| S61I+Y212K;                                                                                  |                                                                                                                                                                                                                                                                                            |
| S75A+H143T+Y212K+Q236L;                                                                      |                                                                                                                                                                                                                                                                                            |
| T65K+T88N+Y212K                                                                              |                                                                                                                                                                                                                                                                                            |
| S29P+F223L;                                                                                  | These variants combine surface stability/electrostatics (29/41/61/65/88/143/212/236) with a pocket-gate change at 223, giving modest net gains and a tendency to favor peroxygenation over strong peroxidation boosts.                                                                     |
| S41A+S75A+T88N+Y212K;                                                                        |                                                                                                                                                                                                                                                                                            |
| S41A+S75A+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+H143T+Y212K;                                                                       |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+S75A+H143T;                                                                        |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+Y212K+Q236L;                                                                       |                                                                                                                                                                                                                                                                                            |
| S41A+T88N+Y212K;                                                                             |                                                                                                                                                                                                                                                                                            |
| S41A+Y212K+Q236L;                                                                            |                                                                                                                                                                                                                                                                                            |
| S61I+T65K+S75A+Y212K;                                                                        |                                                                                                                                                                                                                                                                                            |
| S61I+T88N+H143T+Y212K;                                                                       |                                                                                                                                                                                                                                                                                            |
| T65K+Q236L                                                                                   |                                                                                                                                                                                                                                                                                            |
| H143Q+F220L;                                                                                 | H143Q/T paired with F220L combines a surface electrostatic change with loss of aromatic pocket packing, consistent with reduced productive binding for peroxygenation while maintaining or enhancing peroxidation via easier access.                                                       |
| H143T+F220L                                                                                  |                                                                                                                                                                                                                                                                                            |
| L38M+I64L+S182V;                                                                             | These combinations mix pocket reshaping (38/64/70/182/220) with surface/loop changes (61/67/88/212/236/217), yielding a profile consistent with enhanced access and peroxidation without a matching improvement in peroxygenation.                                                         |
| L38M+T67A+S182V;                                                                             |                                                                                                                                                                                                                                                                                            |
| M70F+S217P+F220L;                                                                            |                                                                                                                                                                                                                                                                                            |
| S61I+T88N+Y212K+Q236L                                                                        |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+Y212K;                                                                             | Adding strong surface electrostatic changes (65/212 with optional 41/75/143) yields only modest net gains and, in this subset, a shift away from peroxidation, suggesting non-additive electrostatic effects.                                                                              |
| T65K+S75A+H143T+Y212K                                                                        |                                                                                                                                                                                                                                                                                            |
| S29P+I64L+T65K+A171F+S182M+Y212K;                                                            | This high-performing cluster couples surface/expression drivers (29/65/88/143/212/236) with distal pocket remodeling (171/182 and sometimes 64/75), consistent with synergistic improvements in enzyme availability and substrate positioning for peroxygenation.                          |
| T65K+S75A+H143T+Y212K+Q236L;                                                                 |                                                                                                                                                                                                                                                                                            |
| T67A+T88N+H143T+A171F+S182M+Y212K                                                            |                                                                                                                                                                                                                                                                                            |
| S29P+T67A+T88N+H143T+A171F+S182M+H208D                                                       | S29P+T67A+T88N+H143T+A171F+S182M+H208D stacks multiple surface electrostatic/dynamic changes with distal pocket remodeling, yielding a profile consistent with peroxygenation-biased selectivity.                                                                                          |
| S41A+G57A+T65K+A171F+S182M+Q236L                                                             | S41A+G57A+T65K+A171F+S182M+Q236L shows strong activity but extremely low yield, indicating a tradeoff where catalytic improvements are offset by poor producibility.                                                                                                                       |
| S29P+S41A+T67A+T88N+A171F+Q236L                                                              | S29P+S41A+T67A+T88N+A171F+Q236L combines mostly surface stabilizers with one distal pocket edit, consistent with improved peroxygenation without a corresponding increase in peroxidation.                                                                                                 |
| T65K+S75A+T88N+A171F+Y212K+H208D+Q236L;                                                      | These variants combine surface electrostatics/stability (65/88/212/236 with 143 and/or 208) with pocket gating (75) and distal pocket remodeling (171), consistent with improved overall turnover and a more balanced peroxygenation/peroxidation profile.                                 |
| T65K+S75A+T88N+H143T+Y212K                                                                   |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+T88N+Y212K+Q236L                                                                   | S41A+T65K+T88N+Y212K+Q236L is surface-focused and lacks direct pocket edits, consistent with a global activity uplift driven by folding/solubility and long-range electrostatics.                                                                                                          |
| S41A+T65K+H143T+Y212K+Q236L                                                                  | S41A+T65K+H143T+Y212K+Q236L adds an additional electrostatic perturbation at 143 to a surface-stabilized background, consistent with a small peroxygenation gain but reduced peroxidation in purified measurements.                                                                        |
| S29P+S41A+T65K+S75A+T88N+A171F+Y212K+S182M;                                                  | When distal pocket remodeling (171/182) is combined with multiple surface stabilizers and a pocket gate (75), peroxygenation can become very high while peroxidation remains moderate, consistent with improved substrate pre-organization.                                                |
| S41A+T65K+T88N+H143T+Y212K                                                                   |                                                                                                                                                                                                                                                                                            |
| G57A+I64L+T65K+H143T+A171F+H208D;                                                            | This broad cluster repeatedly combines surface stability/expression boosters (29/41/57/65/88/143/212/236) with distal pocket remodeling (171/182) and occasional electrostatic tuning (208), yielding consistently strong peroxygenation with moderate peroxidation.                       |
| G57A+I64L+T65K+T88N+H143T+A171F+H208D+Y212K;                                                 |                                                                                                                                                                                                                                                                                            |
| G57A+I64L+T88N+H143T+A171F+S182M+Y212K;                                                      |                                                                                                                                                                                                                                                                                            |
| G57A+T65K+T67A+H143T+A171F+H208D+Q236L;                                                      |                                                                                                                                                                                                                                                                                            |
| G57A+T65K+T67A+T88N+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| I64L+T65K+T67A+T88N+A171F+S182M+Y212K;                                                       |                                                                                                                                                                                                                                                                                            |
| S29P+G57A+I64L+T65K+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| S29P+G57A+I64L+T88N+H143T+A171F+S182M+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| S29P+G57A+T67A+T88N+H143T+A171F+S182M+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T67A+A171F+H208D+Y212K;                                                  |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+T67A+T88N+A171F+S182M+H208D;                                                  |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+T65K+T67A+T88N+H143T+A171F+Y212K;                                                  |                                                                                                                                                                                                                                                                                            |
| S29P+T65K+T67A+T88N+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| S41A+I64L+T65K+T88N+H143T+A171F+S182M+H208D;                                                 |                                                                                                                                                                                                                                                                                            |
| S41A+I64L+T65K+T88N+H143T+A171F+Y212K+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| S41A+I64L+T88N+A171F+Y212K;                                                                  |                                                                                                                                                                                                                                                                                            |
| S61I+T65K+H143T+Y212K+Q236L;                                                                 |                                                                                                                                                                                                                                                                                            |
| S75A+T88N+H143T+Y212K+Q236L;                                                                 |                                                                                                                                                                                                                                                                                            |
| T65K+S75A+T88N+A171F+Y212K+Q236L;                                                            |                                                                                                                                                                                                                                                                                            |
| T65K+S75A+T88N+Y212K+Q236L;                                                                  |                                                                                                                                                                                                                                                                                            |
| T65K+T67A+T88N+H143T+A171F+H208D+Y212K;                                                      |                                                                                                                                                                                                                                                                                            |
| T65K+T67A+T88N+H143T+A171F+Q236L                                                             |                                                                                                                                                                                                                                                                                            |
| S41A+G57A+H143T+A171F+S182M+H208D                                                            | S41A+G57A+H143T+A171F+S182M+H208D combines surface electrostatic changes with distal pocket remodeling and shows a peroxygenation-biased shift with suppressed peroxidation.                                                                                                               |
| S29P+G57A+T88N+H143T+A171F+H208D+Y212K+Q236L;                                                | These multi-mutants combine extensive surface stabilization with distal pocket remodeling (171/182) and limited pocket edits (64/75), generally favoring peroxygenation while damping peroxidation.                                                                                        |
| S29P+S41A+G57A+I64L+A171F+Y212K;                                                             |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+T67A+H143T+A171F+Y212K;                                                       |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+I64L+A171F+Y212K;                                                                  |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+I64L+T88N+A171F+S182M+H208D+Q236L;                                                 |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+T67A+T88N+H143T+A171F+S182M+H208D;                                                 |                                                                                                                                                                                                                                                                                            |
| S29P+T88N+H143T+A171F+S182M+Y212K;                                                           |                                                                                                                                                                                                                                                                                            |
| S41A+G57A+I64L+T67A+T88N+H143T+A171F+Q236L;                                                  |                                                                                                                                                                                                                                                                                            |
| S41A+G57A+I64L+T88N+H143T+A171F+H208D;                                                       |                                                                                                                                                                                                                                                                                            |
| S41A+G57A+T67A+H143T+A171F+Y212K;                                                            |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+S75A+H143T+Y212K;                                                                  |                                                                                                                                                                                                                                                                                            |
| S41A+T65K+T88N+H143T+Y212K+Q236L;                                                            |                                                                                                                                                                                                                                                                                            |
| S41A+T67A+H143T+A171F+Q236L;                                                                 |                                                                                                                                                                                                                                                                                            |
| T65K+S75A+T88N+H143T+A171F+Y212K+Q236L                                                       |                                                                                                                                                                                                                                                                                            |
| S41A+S75A+H143T+Y212K+Q236L                                                                  | S41A+S75A+H143T+Y212K+Q236L shows near-baseline performance, consistent with negative epistasis where pocket gating and surface electrostatics partially cancel benefits (single data point).                                                                                              |
| S41A+G57A+I64L+T65K+T67A+T88N+A171F+S182M+H208D+Q236L                                        | S41A+G57A+I64L+T65K+T67A+T88N+A171F+S182M+H208D+Q236L achieves very high activity but very low yield, indicating strong catalytic synergy coupled to a major expression/folding penalty.                                                                                                   |
| S29P+S41A+G57A+H143T+A167E+A171F+S182M+H208D+Q236L                                           | Adding A167E into an already stacked background strongly suppresses peroxidation while maintaining high peroxygenation, consistent with surface electrostatic tuning that disfavors one-electron pathways.                                                                                 |
| S29P+S41A+T65K+T67A+T88N+A171F+H208D+Y212K+Q236L                                             | S29P+S41A+T65K+T67A+T88N+A171F+H208D+Y212K+Q236L yields high peroxygenation but low peroxidation and low yield, consistent with a peroxygenation-biased but less producible design.                                                                                                        |
| S29P+S41A+G57A+T65K+T88N+H143T+A171F+S182M+Y212K+Q236L                                       | S29P+S41A+G57A+T65K+T88N+H143T+A171F+S182M+Y212K+Q236L combines surface stabilization with distal pocket remodeling to give strong peroxygenation with near-neutral peroxidation and good yield.                                                                                           |
| S29P+S41A+G57A+I64L+T65K+T67A+H143T+A171F+S182M+H208D+Y212K+Q236L                            | This variant shows very high apparent activity but very low purified activities alongside extremely high yield, suggesting assay-context artifacts or a large inactive protein fraction, so mechanistic interpretation is uncertain.                                                       |
| S29P+S41A+I64L+T65K+S75A+T88N+H143T+A171F+S182M+Y212K+Q236L                                  | S29P+S41A+I64L+T65K+S75A+T88N+H143T+A171F+S182M+Y212K+Q236L shows the strongest peroxygenation gain with elevated peroxidation, consistent with synergistic pocket gating plus broad surface stabilization.                                                                                |
| G57A+I64L+T67A+T88N+H143T+A171F+S182M+Y212K+Q236L;                                           | Across these highly stacked mutants, the shared net effect is strong surface stabilization/expression tuning plus distal pocket remodeling that drives high peroxygenation, while additional C-terminal charge/packing edits likely modulate solubility rather than active-site chemistry. |
| I64L+T67A+H143T+A171F+H208D+Y212K+Q236L+S237V+R239E+A240Q+I241S+E242S+L243C;                 |                                                                                                                                                                                                                                                                                            |
| S29P+G57A+I64L+T65K+T67A+H143T+A171F+S182M+Y212K;                                            |                                                                                                                                                                                                                                                                                            |
| S29P+G57A+I64L+T65K+T67A+T88N+H143T+A171F+Y212K+Q236L;                                       |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T65K+H143T+A171F+S182M+H208D;                                            |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T65K+T66M+T67A+A171F+S182M+Y212K+Q236L;                                  |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T65K+T67A+A171F+S182M+H208D+Q236L;                                       |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T88N+A171F+S182M+Y212K+Q236L;                                            |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T88N+H143T+A171F+S182M+Y212K;                                            |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57A+I64L+T88N+H143T+A171F+S182M+Y212K+Q236L;                                      |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+G57L+S61F+I64L+T65K+T67A+S75A+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L;            |                                                                                                                                                                                                                                                                                            |
| S29P+T65K+T67A+T88N+H143T+A171F+S182M+E197K+H208D+Y212K+Q236L;                               |                                                                                                                                                                                                                                                                                            |
| S41A+I64L+T65K+S75A+T88N+H143T+A171F+H208D+Y212K+Q236L;                                      |                                                                                                                                                                                                                                                                                            |
| S41A+I64L+T65K+S75A+T88N+H143T+A171F+Y212K+Q236L                                             |                                                                                                                                                                                                                                                                                            |
| S29P+S41A+S61F+I64L+T65K+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L                            | Adding S61F into a heavily stacked background preserves peroxygenation but reduces peroxidation, consistent with a packing change that alters global dynamics or expression in a non-additive way.                                                                                         |
| S29P+S41A+G57L+I64L+T65K+S75A+T88N+H143T+A171F+H208D+S182M+Y212K+Q236L;                      | These variants combine extensive surface stabilization with distal pocket remodeling and additional C-terminal electrostatic/packing edits, yielding moderate peroxygenation with suppressed peroxidation consistent with a more selective but not maximally active profile.               |
| S41A+G57A+I64L+T67A+H143T+A171F+S182M+H208D+Y212K+Q236L+S237V+R239E+A240Q+I241S+E242S+L243C; |                                                                                                                                                                                                                                                                                            |
| S41A+I64L+T65K+S75R+T88N+H143T+A171F+Y212K+Q236L                                             |                                                                                                                                                                                                                                                                                            |

{'refined_csv_path': '/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/15_analyze_mutants/mutant_effect_explanations_revised.csv',
 'refined_llm_summary_path': '/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/15_analyze_mutants/mutant_analysis_llm_summary.md'}